## GBFS

In [4]:
import heapq

def manhattan_dist(node, goal):
    return abs(node[0] - goal[0]) + abs(node[1] - goal[1])

def gbfs(maze,start,goal):

    directions = [(-1,0),(1,0),(0,-1),(0,1)]
    frontier = []
    heapq.heappush(frontier,(manhattan_dist(start,goal),start))
    visited=set()
    visited.add(start)
    parent = {start:None}
    nodes_expanded = 0

    while frontier:
        _,current = heapq.heappop(frontier)

        nodes_expanded+=1

        if current == goal:
            break

        for move in directions:
            next_node = (current[0]+move[0], current[1]+move[1])


            if(0<= next_node[0] < len(maze) and
               0<= next_node[1] < len(maze[0]) and
               maze[next_node[0]][next_node[1]] == 0 and
              next_node not in visited):

                visited.add(next_node)

                parent[next_node] = current
                h = manhattan_dist(next_node, goal)
                heapq.heappush(frontier,(h,next_node))


    path = []
    node=goal
    while node is not None:
        path.append(node)
        node = parent.get(node)

    path.reverse()

    if path[0] == start:
        return path, nodes_expanded
    else:
        return None, nodes_expanded

maze = [
    [0, 0, 0, 1, 0],
    [1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0]
]

start=(0,0)
goal = (4,4)

path, nodes_expanded = gbfs(maze,start,goal)

if path:
    print("path found:")
    print(path)
else:
    print("no path found")

print("no of nodes expanded", nodes_expanded)

path found:
[(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]
no of nodes expanded 9


## HILL CLIMBING


In [ ]:
import numpy as np
import random
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

data = load_iris()
X = data.data
y = data.target
n_features = X.shape[1]

def evaluate(state):
    if sum(state) == 0:
        return 0

    selected_features = [i for i in range(len(state)) if state[i] == 1]
    X_subset = X[:, selected_features]

    model = LogisticRegression(max_iter=200)
    scores = cross_val_score(model, X_subset, y, cv=5)

    return scores.mean()

def generate_neighbors(state):
    neighbors = []

    for i in range(len(state)):
        neighbor = state.copy()
        neighbor[i] = 1 - neighbor[i]
        neighbors.append(neighbor)

    return neighbors

def hill_climbing():
    current_state = [random.choice([0, 1]) for _ in range(n_features)]
    current_score = evaluate(current_state)

    while True:
        neighbors = generate_neighbors(current_state)
        neighbor_scores = [(neighbor, evaluate(neighbor)) for neighbor in neighbors]

        best_neighbor, best_score = max(neighbor_scores, key=lambda x: x[1])

        if best_score <= current_score:
            break

        current_state = best_neighbor
        current_score = best_score

    return current_state, current_score

if __name__ == "__main__":
    runs = 5

    for i in range(runs):
        best_state, best_score = hill_climbing()

        print(f"Run {i+1}:")
        print("Selected Features:", best_state)
        print("Accuracy:", round(best_score, 4))
        print("-" * 40)

WEEK8

In [ ]:
class MedicalEnvironment:
    def __init__(self, fever=False, cough=False, chest_pain=False):
        self.symptoms = {
            "Fever": fever,
            "Cough": cough,
            "ChestPain": chest_pain
        }

    def get_percepts(self):
        return self.symptoms

    def display(self):
        print("Patient Symptoms:")
        for k, v in self.symptoms.items():
            print(f"  {k}: {v}")
        print("-" * 40)


class MedicalAgent:
    def __init__(self):
        self.rules = [ 
            (["Fever", "Cough"], "Flu"),
            (["Fever", "Cough", "ChestPain"], "Pneumonia"),
            (["Pneumonia"], "HospitalizationRequired")
        ]

        self.facts = set()

    def update_percepts(self, percepts):
        for symptom, present in percepts.items():
            if present:
                self.facts.add(symptom)

    def forward_chain(self):
        inferred = True

        while inferred:
            inferred = False

            for conditions, conclusion in self.rules:
                if all(cond in self.facts for cond in conditions):
                    if conclusion not in self.facts:
                        self.facts.add(conclusion)
                        inferred = True

    def get_results(self):
        diseases = []
        actions = []

        if "Flu" in self.facts:
            diseases.append("Flu")

        if "Pneumonia" in self.facts:
            diseases.append("Pneumonia")

        if "HospitalizationRequired" in self.facts:
            actions.append("Hospitalization Required")

        return diseases, actions

    def act(self, percepts):
        self.update_percepts(percepts)
        self.forward_chain()
        return self.get_results()

In [ ]:
env = MedicalEnvironment(fever=True, cough=True, chest_pain=True)
agent = MedicalAgent()

env.display()

percepts = env.get_percepts()
diseases, actions = agent.act(percepts)

print("Inferred Diseases:", diseases)
print("Recommended Actions:", actions)